# NeuralMed — Deliverable 3: Extended Evaluation & Refinements

**Author:** Honey Reddy Daram  
**Course:** Deep Learning — University of Florida  
**Date:** April 2026

This notebook documents all D3 improvements:
1. 5-fold stratified cross-validation (addresses D2 overfitting)
2. Confusion matrices + ROC curves for all 5 tabular models
3. D2 vs D3 accuracy comparison
4. Feature importance analysis
5. Model stability analysis (CV box plots)


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, json

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.datasets import load_breast_cancer

os.makedirs('../results', exist_ok=True)
print("Libraries loaded successfully")


## 1. Data Loading
All datasets sourced from UCI / Kaggle (same as D2). Datasets are in `data/` after running `python -c 'from sklearn.datasets import fetch_openml; ...'`.

In [ ]:
def prep_diabetes():
    df = pd.read_csv('../data/diabetes.csv')
    X = df.drop('Outcome', axis=1).values.astype(float)
    y = df['Outcome'].values.astype(int)
    return X, y, list(df.drop('Outcome', axis=1).columns)

def prep_heart():
    df = pd.read_csv('../data/heart.csv')
    for c in df.select_dtypes(['object']).columns:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))
    df = df.fillna(df.median(numeric_only=True))
    target = df.columns[-1]
    X = df.drop(target, axis=1).values.astype(float)
    y = LabelEncoder().fit_transform(df[target].values)
    return X, y, list(df.drop(target, axis=1).columns)

def prep_parkinsons():
    df = pd.read_csv('../data/parkinsons.csv')
    feature_cols = [c for c in df.columns if c.startswith('V')][:8]
    target = 'Class' if 'Class' in df.columns else df.columns[-1]
    X = df[feature_cols].values.astype(float)
    y = LabelEncoder().fit_transform(df[target].values)
    feat_names = ['MDVP:Fo','MDVP:Fhi','MDVP:Flo','MDVP:Jitter','RPDE','DFA','spread2','D2']
    return X, y, feat_names

def prep_liver():
    df = pd.read_csv('../data/liver.csv')
    for c in df.select_dtypes(['object']).columns:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str))
    df = df.fillna(df.median(numeric_only=True))
    target = df.columns[-1]
    X = df.drop(target, axis=1).values.astype(float)
    y = LabelEncoder().fit_transform(df[target].values)
    return X, y, list(df.drop(target, axis=1).columns)

def prep_bcancer():
    bc = load_breast_cancer()
    return bc.data, bc.target, list(bc.feature_names)

datasets = {
    'Diabetes':     prep_diabetes(),
    'Heart Disease': prep_heart(),
    "Parkinson's":  prep_parkinsons(),
    'Liver Disease': prep_liver(),
    'Breast Cancer': prep_bcancer(),
}
for n, (X, y, f) in datasets.items():
    print(f"{n:20s}  shape={X.shape}  classes={len(set(y))}")


## 2. Model Registry
D3 key change: `max_depth=10` on all tree-based models to prevent overfitting (D2 concern).

In [ ]:
def get_models():
    return {
        'Naive Bayes':    GaussianNB(),
        'Logistic Reg.':  LogisticRegression(max_iter=1000, random_state=42),
        'KNN':            KNeighborsClassifier(n_neighbors=5),
        'SVM':            SVC(probability=True, random_state=42),
        'Decision Tree':  DecisionTreeClassifier(max_depth=8, random_state=42),   # constrained
        'Random Forest':  RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),  # constrained
        'XGBoost':        XGBClassifier(n_estimators=100, max_depth=5, eval_metric='logloss',
                                        random_state=42, verbosity=0),            # constrained
        'Extra Trees':    ExtraTreesClassifier(n_estimators=100, max_depth=10, random_state=42),    # constrained
    }
print("D3 model registry: all tree-based models have max_depth constraints")


## 3. Full Metrics Table — 5-Fold Cross-Validation + Hold-Out Test
D3 uses stratified 5-fold CV to replace the single-split evaluation from D2.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
summary = {}

for dname, (X, y, feats) in datasets.items():
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.2, stratify=y, random_state=42)
    summary[dname] = {}
    for mname, model in get_models().items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        cv_scores = cross_val_score(model, X_s, y, cv=cv, scoring='accuracy')
        summary[dname][mname] = {
            'Test Acc':  round(accuracy_score(y_te, y_pred)*100, 1),
            'Precision': round(precision_score(y_te, y_pred, average='weighted', zero_division=0)*100, 1),
            'Recall':    round(recall_score(y_te, y_pred, average='weighted')*100, 1),
            'F1':        round(f1_score(y_te, y_pred, average='weighted')*100, 1),
            'CV Mean':   round(cv_scores.mean()*100, 1),
            'CV Std':    round(cv_scores.std()*100, 1),
        }

for dname, mres in summary.items():
    print(f"\n{'='*60}")
    print(f"  {dname}")
    print(f"{'='*60}")
    df_table = pd.DataFrame(mres).T
    print(df_table.to_string())


## 4. Cross-Validation Bar Chart — All Models

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5.5))
BG='#050b14'; SURF='#0a1628'; CYAN='#00e5ff'; TEXT='#e2eaf4'; BORDER='#1e3a5f'; AMBER='#f0b429'
mcolors = [CYAN,'#4cc9f0','#00e096','#f72585',AMBER,'#7b2d8b','#e67e22','#2ecc71']

for ax, (dname, mres) in zip(axes, summary.items()):
    names = list(mres.keys())
    means = [mres[n]['CV Mean'] for n in names]
    stds  = [mres[n]['CV Std']  for n in names]
    bars = ax.barh(names, means, xerr=stds, color=mcolors, alpha=0.88, capsize=3,
                   error_kw={'ecolor':TEXT,'capthick':1.2})
    ax.set_facecolor(SURF); ax.set_xlim(40, 115)
    ax.set_title(dname, color=CYAN, fontweight='bold')
    ax.set_xlabel('CV Accuracy (%)', color=TEXT)
    ax.spines[:].set_color(BORDER); ax.axvline(80, color=AMBER, ls='--', lw=0.8, alpha=0.5)
    for bar, m in zip(bars, means):
        ax.text(min(m+0.5, 113), bar.get_y()+bar.get_height()/2, f'{m:.1f}', va='center', fontsize=8)

fig.patch.set_facecolor(BG)
fig.suptitle('5-Fold Cross-Validation Accuracy — All 8 Models × 5 Diseases (D3)', 
             color=TEXT, fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/cv_all_models.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


## 5. Confusion Matrices — Best Model per Disease

In [ ]:
# Find best model per disease
best_models = {}
for dname, (X, y, feats) in datasets.items():
    scaler = StandardScaler(); X_s = scaler.fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.2, stratify=y, random_state=42)
    best_acc, best_obj, best_name = 0, None, ''
    for mname, model in get_models().items():
        model.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, model.predict(X_te))
        if acc > best_acc:
            best_acc, best_obj, best_name = acc, model, mname
    best_models[dname] = (best_obj, X_tr, X_te, y_tr, y_te, feats, scaler, best_name, best_acc)

labels_map = {
    'Diabetes':     ['No Diabetes','Diabetes'],
    'Heart Disease':['Healthy','Disease'],
    "Parkinson's":  ['Healthy',"Parkinson's"],
    'Liver Disease':['Healthy','Disease'],
    'Breast Cancer':['Malignant','Benign'],
}
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
fig.patch.set_facecolor(BG)
for ax, (dname, (model, X_tr, X_te, y_tr, y_te, feats, sc, mname, acc)) in zip(axes, best_models.items()):
    cm = confusion_matrix(y_te, model.predict(X_te))
    lbs = labels_map.get(dname, ['0','1'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=lbs, yticklabels=lbs, linewidths=0.5,
                annot_kws={'size':12,'weight':'bold'})
    ax.set_title(f'{dname}\n{mname} ({acc*100:.1f}%)', color=CYAN, fontweight='bold', fontsize=9)
    ax.set_xlabel('Predicted', color=TEXT); ax.set_ylabel('Actual', color=TEXT)

fig.suptitle('Confusion Matrices — Best Model per Disease (D3)', color=TEXT, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


## 6. ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
fig.patch.set_facecolor(BG)
MUTED='#6b8299'
roc_colors_list = [CYAN, '#00e096', AMBER, '#ff4f6d', '#9b59b6']

for ax, ((dname,(model,X_tr,X_te,y_tr,y_te,feats,sc,mname,acc)),col) in zip(axes, zip(best_models.items(), roc_colors_list)):
    y_prob = model.predict_proba(X_te)[:,1]
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.set_facecolor(SURF)
    ax.plot(fpr, tpr, color=col, lw=2.2)
    ax.plot([0,1],[0,1], color=MUTED, ls='--', lw=1)
    ax.fill_between(fpr, tpr, alpha=0.12, color=col)
    ax.set_title(dname, color=col, fontweight='bold', fontsize=10)
    ax.set_xlabel('FPR', color=TEXT); ax.set_ylabel('TPR', color=TEXT)
    ax.text(0.42, 0.08, f'AUC={roc_auc:.3f}', transform=ax.transAxes, color=TEXT,
            fontsize=11, fontweight='bold', bbox=dict(facecolor=SURF, edgecolor=col, boxstyle='round,pad=0.3'))
    ax.spines[:].set_color(BORDER); ax.set_xlim([-0.02,1.02]); ax.set_ylim([-0.02,1.05])
    ax.grid(True, alpha=0.2)

fig.suptitle('ROC Curves — Best Model per Disease (D3)', color=TEXT, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/roc_curves.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


## 7. D2 vs D3 Accuracy Comparison

In [ ]:
d2_acc = {
    'Diabetes': 84.5, 'Heart Disease': 92.2,
    "Parkinson's": 92.3, 'Liver Disease': 72.0, 'Breast Cancer': 95.0
}
# D2 note: RF/DT achieved 100% — suspected overfitting; SVM/best reliable selected
d3_acc = {d: best_models[d][-1]*100 for d in best_models}

fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor(BG); ax.set_facecolor(SURF)
x = np.arange(5); w = 0.35
b1 = ax.bar(x-w/2, [d2_acc[d] for d in d2_acc], w, label='D2 (single split, unconstrained)', color=AMBER, alpha=0.85)
b2 = ax.bar(x+w/2, [d3_acc.get(d,0) for d in d2_acc], w, label='D3 (max_depth + 5-fold CV)', color=CYAN, alpha=0.85)
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{b.get_height():.1f}%',
            ha='center', va='bottom', fontsize=9, color=TEXT)
ax.set_xticks(x); ax.set_xticklabels(list(d2_acc.keys()), color=TEXT, fontsize=11)
ax.set_ylim(50, 112); ax.set_ylabel('Test Accuracy (%)', color=TEXT)
ax.set_title('D2 vs D3: Accuracy Comparison\n(D3 accuracy is honest — overfitting addressed with depth constraints + CV)', 
             color=TEXT, fontweight='bold', fontsize=12)
ax.legend(facecolor=SURF, edgecolor=BORDER, labelcolor=TEXT, fontsize=10)
ax.spines[:].set_color(BORDER); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/d2_vs_d3_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("\nNote: Some D3 accuracies are lower than D2 because we corrected overfitting.")
print("D2 RF on heart achieved 100% — this was a known artifact. D3 uses constrained models.")


## 8. Feature Importance Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
fig.patch.set_facecolor(BG)
for ax, (dname, col) in zip(axes, [("Parkinson's",AMBER),('Liver Disease','#00e096'),('Breast Cancer',CYAN)]):
    model_obj, _, _, _, _, feats, _, mname, _ = best_models[dname]
    if hasattr(model_obj, 'feature_importances_'):
        imps = model_obj.feature_importances_
        top_n = min(10, len(feats))
        idx = np.argsort(imps)[-top_n:]
        ax.barh([str(feats[i])[:22] for i in idx], imps[idx]*100, color=col, alpha=0.85)
        ax.set_xlabel('Importance (%)', color=TEXT)
    elif hasattr(model_obj, 'coef_'):
        coefs = np.abs(model_obj.coef_[0])
        idx = np.argsort(coefs)[-10:]
        ax.barh([str(feats[i])[:22] for i in idx], coefs[idx], color=col, alpha=0.85)
        ax.set_xlabel('|Coefficient|', color=TEXT)
    ax.set_facecolor(SURF); ax.spines[:].set_color(BORDER)
    ax.set_title(f'{dname} ({mname})', color=col, fontweight='bold', fontsize=10)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('Top-10 Feature Importances — D3 Models', color=TEXT, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()


## 9. Summary — All D3 Results

In [ ]:
print("=" * 75)
print(f"{'Disease':20s}  {'Best Model':18s}  {'TestAcc':>8s}  {'F1':>6s}  {'CV Mean±Std':>14s}")
print("=" * 75)
for dname, mres in summary.items():
    best_m = max(mres.items(), key=lambda x: x[1]['Test Acc'])
    mn, met = best_m
    print(f"{dname:20s}  {mn:18s}  {met['Test Acc']:>7.1f}%  {met['F1']:>5.1f}%  {met['CV Mean']:>5.1f}±{met['CV Std']:.1f}%")
print()
print("D2 overfitting addressed: Decision Tree / RF now constrained with max_depth")
print("5-fold CV gives robust estimates; CV std quantifies dataset-size variance")
